In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# 베이스 상태 인코딩
# runners: 비트마스크 (1루=1, 2루=2, 3루=4)
# ============================================================

def advance_runners(runners, outs, event):
    """
    event: 'single','double','triple','hr','bb','k','out'
    returns: (new_runners, new_outs, runs_scored)
    """
    runs = 0
    r1 = bool(runners & 1)
    r2 = bool(runners & 2)
    r3 = bool(runners & 4)

    if event == 'hr':
        runs = 1 + r1 + r2 + r3
        return 0, outs, runs

    elif event == 'triple':
        runs = r1 + r2 + r3
        return 4, outs, runs

    elif event == 'double':
        runs = r2 + r3
        new_r2 = 2
        new_r3 = 4 if r1 else 0
        return new_r2 | new_r3, outs, runs

    elif event == 'single':
        runs = r2 + r3
        new_r1 = 1
        new_r2 = 2 if r1 else 0
        return new_r1 | new_r2, outs, runs

    elif event == 'bb':
        if r1 and r2 and r3:
            return 7, outs, 1
        elif r1 and r2:
            return 7, outs, 0
        elif r1:
            return 3, outs, 0
        else:
            return runners | 1, outs, 0

    elif event in ('k', 'out'):
        return runners, outs + 1, 0

    return runners, outs, 0


In [7]:
def simulate_inning(batter_probs_list, lineup_start):
    """
    이닝 시뮬 — lineup_start 위치부터 타순 순환
    returns: (runs_scored, next_lineup_idx)
    """
    if not batter_probs_list:
        raise ValueError("lineup_probs가 비어 있습니다. 타순 구성 셀을 먼저 실행하세요.")
    runners, outs, total_runs = 0, 0, 0
    idx = lineup_start
    n   = len(batter_probs_list)

    while outs < 3:
        probs = batter_probs_list[idx % n]
        event = np.random.choice(list(probs.keys()), p=list(probs.values()))
        runners, outs, runs = advance_runners(runners, outs, event)
        total_runs += runs
        idx += 1

    return total_runs, idx % n


def simulate_game(batter_probs_list, n_innings=9):
    """
    9이닝 경기 시뮬 — 타순이 이닝을 넘어 연속 순환
    returns: 팀 득점
    """
    total_runs = 0
    lineup_idx = 0

    for _ in range(n_innings):
        runs, lineup_idx = simulate_inning(batter_probs_list, lineup_idx)
        total_runs += runs

    return total_runs


def simulate_opponent_runs(pitcher_era=4.50, n_innings=9):
    """
    상대 득점 시뮬 — Rangers 투수 ERA 기반 Poisson 근사
    ERA 9 = 9이닝 기대 실점
    """
    expected_runs = pitcher_era * n_innings / 9
    return int(np.random.poisson(expected_runs))


def simulate_season(batter_probs_list, n_games=162,
                    team_era=4.00, n_sims=1000):
    """
    시즌 시뮬 n_sims회 반복 → 평균 승패 반환
    team_era: Rangers 투수 ERA (상대 득점 산출)
    """
    season_wins = []

    for _ in range(n_sims):
        wins = 0
        for _ in range(n_games):
            rangers_runs  = simulate_game(batter_probs_list)
            opponent_runs = simulate_opponent_runs(team_era)
            if rangers_runs > opponent_runs:
                wins += 1
        season_wins.append(wins)

    arr = np.array(season_wins)
    print(f"시즌 시뮬 {n_sims}회 결과")
    print(f"  평균 승: {arr.mean():.1f}  |  중앙값: {np.median(arr):.0f}")
    print(f"  5th~95th percentile: {np.percentile(arr, 5):.0f} ~ {np.percentile(arr, 95):.0f}승")
    return arr


In [8]:
def extract_probs(row):
    """
    타자 데이터 1행 → 이벤트 확률 딕셔너리
    """
    pa        = row["PA"]
    h         = row["H"]
    double_   = row["2B"]
    triple_   = row["3B"]
    hr        = row["HR"]
    bb        = row["BB"]
    strikeout = row["SO"]

    single    = (h - double_ - triple_ - hr) / pa
    out       = max(0, 1 - (h + bb + strikeout) / pa)

    probs = {
        'single': single,
        'double': double_ / pa,
        'triple': triple_ / pa,
        'hr':     hr       / pa,
        'bb':     bb       / pa,
        'k':      strikeout / pa,
        'out':    out,
    }

    # 음수 방지 + 합계 1.0 보정
    probs = {key: max(0, val) for key, val in probs.items()}
    total = sum(probs.values())
    return {key: val / total for key, val in probs.items()}


In [9]:
hit_stat = pd.read_csv("./data/rangers_hitting_statcast.csv")

# Rangers 주전 타순
lineup = [
    "Marcus Semien",
    "Corey Seager",
    "Wyatt Langford",
    "Adolis García",
    "Jake Burger",
    "Josh Jung",
    "Jonah Heim",
    "Josh Smith",
    "Kyle Higashioka",
]

lineup_probs = []
for name in lineup:
    row = hit_stat[hit_stat["Player"] == name]
    if row.empty:
        print(f"⚠️  {name} 데이터 없음")
        continue
    lineup_probs.append(extract_probs(row.iloc[0]))

print(f"타순 구성: {len(lineup_probs)}명")
for name, probs in zip(lineup, lineup_probs):
    print(f"  {name:<22}  HR={probs['hr']:.3f}  BB={probs['bb']:.3f}  K={probs['k']:.3f}")


타순 구성: 9명
  Marcus Semien           HR=0.028  BB=0.094  K=0.174
  Corey Seager            HR=0.047  BB=0.130  K=0.196
  Wyatt Langford          HR=0.038  BB=0.129  K=0.264
  Adolis García           HR=0.035  BB=0.051  K=0.247
  Jake Burger             HR=0.043  BB=0.032  K=0.247
  Josh Jung               HR=0.027  BB=0.053  K=0.252
  Jonah Heim              HR=0.025  BB=0.074  K=0.203
  Josh Smith              HR=0.018  BB=0.098  K=0.178
  Kyle Higashioka         HR=0.034  BB=0.061  K=0.220


In [10]:
# ============================================================
# 1경기 시뮬 샘플
# ============================================================
np.random.seed(42)
sample_runs = [simulate_game(lineup_probs) for _ in range(1000)]
print(f"1경기 득점 분포 (1000회):")
print(f"  평균 {np.mean(sample_runs):.2f}점  |  중앙값 {np.median(sample_runs):.0f}점")
print(f"  0점 비율: {sample_runs.count(0)/10:.1f}%  |  5점 이상: {sum(r>=5 for r in sample_runs)/10:.1f}%")

# ============================================================
# 시즌 시뮬 (1000회)
# Rangers 2025 팀 ERA ≈ 4.19 사용
# ============================================================
np.random.seed(0)
wins_dist = simulate_season(lineup_probs, n_games=162, team_era=4.19, n_sims=1000)


1경기 득점 분포 (1000회):
  평균 3.73점  |  중앙값 3점
  0점 비율: 10.0%  |  5점 이상: 34.8%
시즌 시뮬 1000회 결과
  평균 승: 59.3  |  중앙값: 59
  5th~95th percentile: 49 ~ 69승


In [11]:
# ============================================================
# 시나리오 시뮬
# ============================================================

def apply_batter_boost(probs, hr_mult=1.0, bb_mult=1.0, k_mult=1.0, single_mult=1.0):
    """
    특정 타자 probs 조정 후 합계 1.0 재정규화
    hr_mult=1.2   → HR 확률 20% 증가
    k_mult=0.85   → 삼진 15% 감소
    """
    p = probs.copy()
    p['hr']     *= hr_mult
    p['bb']     *= bb_mult
    p['k']      *= k_mult
    p['single'] *= single_mult
    p = {k: max(0, v) for k, v in p.items()}
    total = sum(p.values())
    return {k: v / total for k, v in p.items()}


def build_scenario_lineup(base_lineup_probs, boosts):
    """
    base_lineup_probs: 기존 타순 probs 리스트
    boosts: {타순 인덱스(0-8): dict(hr_mult=, bb_mult=, ...)} 
    returns: 조정된 타순 probs 리스트
    """
    new_lineup = [p.copy() for p in base_lineup_probs]
    for idx, kwargs in boosts.items():
        new_lineup[idx] = apply_batter_boost(new_lineup[idx], **kwargs)
    return new_lineup


def compare_scenarios(scenarios, n_games=162, n_sims=500):
    """
    scenarios: [
        {'name': '기준', 'lineup': lineup_probs, 'team_era': 4.80},
        {'name': '불펜 보강', 'lineup': lineup_probs, 'team_era': 3.80},
        ...
    ]
    """
    results = []
    for sc in scenarios:
        wins_arr = []
        for _ in range(n_sims):
            wins = 0
            for _ in range(n_games):
                r = simulate_game(sc['lineup'])
                o = simulate_opponent_runs(sc['team_era'])
                if r > o:
                    wins += 1
            wins_arr.append(wins)
        arr = np.array(wins_arr)
        results.append({
            'name':    sc['name'],
            'mean':    arr.mean(),
            'median':  np.median(arr),
            'p5':      np.percentile(arr, 5),
            'p95':     np.percentile(arr, 95),
            'era':     sc['team_era'],
        })

    df_res = pd.DataFrame(results)
    df_res['평균승'] = df_res['mean'].round(1)
    df_res['중앙값'] = df_res['median'].astype(int)
    df_res['범위(5~95%)'] = df_res.apply(lambda r: f"{int(r.p5)}~{int(r.p95)}", axis=1)
    df_res['투수ERA'] = df_res['era']
    display(df_res[['name','투수ERA','평균승','중앙값','범위(5~95%)']]
            .rename(columns={'name':'시나리오'}))
    return df_res


In [12]:
np.random.seed(0)

# ── 시나리오 정의 ──────────────────────────────────────────
# 타순 인덱스: 0=Semien 1=Seager 2=Langford 3=Garcia
#              4=Burger  5=Jung   6=Heim    7=Smith  8=Higashioka

# 시나리오 1: 기준 (2025 실적 그대로)
lineup_base = lineup_probs

# 시나리오 2: 불펜 보강만 (ERA 4.80 → 3.80)
lineup_bullpen = lineup_probs

# 시나리오 3: 타자 기량 상승
# Seager 복귀 풀시즌 (HR+20%, BB+15%, K-10%)
# Langford 2년차 (HR+15%, 단타+10%)
# Jung 건강 복귀 (단타+20%, BB+10%)
boosts = {
    1: dict(hr_mult=1.20, bb_mult=1.15, k_mult=0.90),  # Seager
    2: dict(hr_mult=1.15, single_mult=1.10),            # Langford
    5: dict(single_mult=1.20, bb_mult=1.10),            # Jung
}
lineup_batter_up = build_scenario_lineup(lineup_probs, boosts)

# 시나리오 4: 불펜 보강 + 타자 상승 (풀 시나리오)
lineup_full = build_scenario_lineup(lineup_probs, boosts)

scenarios = [
    {'name': '① 기준 (2025)',           'lineup': lineup_base,      'team_era': 4.80},
    {'name': '② 불펜 보강 (ERA↓)',      'lineup': lineup_bullpen,   'team_era': 3.80},
    {'name': '③ 타자 기량 상승',         'lineup': lineup_batter_up, 'team_era': 4.80},
    {'name': '④ 불펜+타자 동시 상승',   'lineup': lineup_full,      'team_era': 3.80},
]

compare_scenarios(scenarios, n_games=162, n_sims=500)


,시나리오,투수ERA,평균승,중앙값,범위(5~95%)
0,① 기준 (2025),4.8,49.9,50,41~59
1,② 불펜 보강 (ERA↓),3.8,65.6,66,56~75
2,③ 타자 기량 상승,4.8,53.9,54,44~63
3,④ 불펜+타자 동시 상승,3.8,69.8,70,60~80


,name,mean,median,p5,p95,era,평균승,중앙값,범위(5~95%),투수ERA
0,① 기준 (2025),49.906,50.0,41.0,59.05,4.8,49.9,50,41~59,4.8
1,② 불펜 보강 (ERA↓),65.600,66.0,56.0,75.00,3.8,65.6,66,56~75,3.8
2,③ 타자 기량 상승,53.890,54.0,44.0,63.05,4.8,53.9,54,44~63,4.8
3,④ 불펜+타자 동시 상승,69.772,70.0,60.0,80.00,3.8,69.8,70,60~80,3.8


## 개선 방향: ML 추가

#### ML 후보
 1. XGBoost/RandomForest — 투구 결과 예측 (Statcast pitch-level data → 타격 이벤트 확률)
 
 2. Neural Net (LSTM) — 선수 시즌 중 성적 변화 예측 (시계열)
 
 3. Hybrid — ML로 타자별 이벤트 확률을 더 정밀하게 추정하고, 그 확률을 지금 Markov Chain에 꽂는 방식

---

#### 딥러닝 후보 
 1. LSTM / Transformer — 선수 성적 시계열 예측

    - 월별 OPS 흐름 → 다음 달 성적 예측
    - 부상 전후 회복 패턴 학습
 
 2. CNN — 투구 궤적 분석

    - Statcast 릴리스포인트 + 무브먼트 → 타격 결과 예측
    - 구질 분류

 3. 강화학습 (RL) — 전략 최적화

    - 번트/도루/교체 타이밍 등 감독 의사결정
    - 타순 최적화

       현재:
              extract_probs() → 단순 PA/AB 비율 계산

       Hybrid로 개선:
              XGBoost/RF → 타자별 이벤트 확률 예측
             (xwOBA, Barrel%, Chase%, EV 등 피처 활용)
               ↓
       Markov Chain에 그 확률 그대로 꽂기

    현재 모델 문제
        → 단순 비율이라 샘플 노이즈 큼
        → 소량 타석 선수 확률 불안정
        → 상황(홈/원정, 좌우투수) 반영 안 됨

    XGBoost 추가 시
        → Statcast 지표로 보정된 확률 사용
        → xwOBA 기반이라 운 제거
        → 특성 중요도로 어떤 지표가 득점에 영향 큰지 파악

## 딥러닝 사용 시

    LSTM → 월별 성적 흐름 있으니까 바로 쓸 수 있음
        부상 전후 패턴 이미 분석했으니 연결 자연스러움
       
    CNN  → 구질 분석용인데 pitch-level 영상 없으면 의미 없음
        Statcast 무브먼트 데이터로는 가능하지만 범위 넓어짐

    RL   → 재미있지만 보상함수 설계가 어렵고
        지금 분석 주제(기대성적 vs 실제)와 거리 있음

## 일단 ml만 추가해 보는 걸로??

    [기존]
        PA/AB/H/2B/3B/HR/BB/SO → 단순 비율 → Markov Chain

    [개선]
        Statcast 피처 → XGBoost → 보정된 확률 → Markov Chain

#### 1단계 (바로 가능) → Hybrid
   XGBoost로 이벤트 확률 개선
   → 현재 49.9승 → 81승 괴리 보정 가능

#### 2단계 (다음)     → LSTM
   월별 게임로그 데이터 있으니까
   → 선수 성적 흐름 예측
   → 부상 전후 패턴 활용

#### 3단계 (나중에)   → RL
   Markov Chain 기반 의사결정 최적화
   → 번트/강공 전략 비교